In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "Quan"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True, errors='ignore')

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Load the Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png'

# Convert the image to base64 for the dashboard
encoded_image = base64.b64encode(open(image_filename, 'rb').read())


app.layout = html.Div([
    html.Div(id='hidden-div', style={'display': 'none'}),
    
    # Display the Grazioso Salvare logo
    html.Center(
        html.Img(
            src='data:image/png;base64,{}'.format(
            encoded_image.decode()
        ),
        style={'width': '180px'}
      )
    ),
    
    # Display the dashboard title
    html.Center(
        html.H1('Grazioso Salvare Rescue Dashboard')
    ),
    
    # Display the student identifier 
    html.Center(
        html.H4('Quavonda Bell - CS 340')
    ),
    
    html.Hr(),
    
    # Interactive rescue type filtering options
    html.Div([
        html.H3('Select a Rescue Type'),
        
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'All Animals (Reset)', 'value': 'Reset'},
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'}
            ],
            value='Reset',
            labelStyle={'display': 'block', 'margin': '8px'}
        )

    ]),
    
    html.Hr(),
    # Display animal records from MongoDB
    dash_table.DataTable(
        id='datatable-id',
        
        # Set up the table columns
        columns=[
            {"name": i, "id": i, "deletable": False,
             "selectable": True}
            for i in df.columns
        ],
        
        # Display all records until a rescue filter is selected
        data=df.to_dict('records'),
        
        # Allow one animal to be selected at a time
        row_selectable='single',
        selected_rows=[0],
        
        # Make the table easier to navigate
        page_size=10,
        page_action='native',
        sort_action='native',
        filter_action='native',
        
        # Keep the table readable
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '100px',
            'whiteSpace': 'normal'
        }
    ),
    
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(
        className='row',
        style={'display' : 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={'width': '50%'}
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
                style={
                    'width': '50%',
                    'minHeight': '500px'
                }
            )
        ]
    )
])
#############################################
# Interaction Between Components / Controller
#############################################



    
# Update the data table based on the selected resuce type
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    
    # Start with an empty query for the Reset option
    query = {}
    
    if filter_type == 'Water':
        query = {
            "animal_type": "Dog",
            "breed": {"$in":[
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland"
            ]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }
        
    elif filter_type == 'Mountain':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler"
            ]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }
     
    elif filter_type == 'Disaster':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Bloodhound",
                "Rottweiler"
            ]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }
        
    # Retreive matching records using the CRUD module
    filtered_df = pd.DataFrame.from_records(db.read(query))
    
    # Remove the MongoDB ID column if it exists
    filtered_df.drop(columns=['_id'], inplace=True, errors='ignore')
    
    # Return the filter records to the dashboard table
    return filtered_df.to_dict('records')

# Update the pie chart when the dashboard data changes
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    # Check whether the table has data to display
    if not viewData:
        return []
    # Convert the filtered table records into a dataframe
    chart_df = pd.DataFrame(viewData)
    
    # Count the animals for each breed
    breed_counts = chart_df['breed'].fillna('Unknown').value_counts()
    
    # Keep the ten most common breeds
    top_breeds = breed_counts.head(10).copy()
    
    # Count the animals in all remaining breeds
    other_count = breed_counts.iloc[10:].sum()
    
    # Combine the remaining breeds into one category
    if other_count > 0:
        top_breeds['Other'] = other_count
        
    # Create a cleaner pie chart
    fig = px.pie(
        names=top_breeds.index,
        values=top_breeds.values,
        title='Animal Breeds by Rescue Type',
        hole=0.3
    )
    
    # Improve chart readability
    fig.update_traces(
        textposition='inside',
        textinfo='percent',
        hovertemplate='%{label}: %{value} animals (%{percent})<extra></extra>'
    )
    
    fig.update_layout(
        height=350,
        margin=dict(l=10, r=10, t=50, b=10),
        legend=dict(font=dict(size=10))
    )
    
    return [dcc.Graph(figure=fig)]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
# Update the map when the table data or selection changes
@app.callback(
    Output('map-id', 'children'),
    [
        Input('datatable-id', 'derived_virtual_data'),
        Input('datatable-id', 'derived_virtual_selected_rows')
    ]
)
def update_map(viewData, index):
    
    # Create the map centered around Austin, Texas
    map_center = [30.75, -97.48]
    
    # Use original records when the table first loads
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_records(viewData)
        
        # Make sure the data contains valid coordinates
        if not dff.empty:
            dff['location_lat'] = pd.to_numeric(
                dff['location_lat'], errors='coerce'
            )
            dff['location_long'] = pd.to_numeric(
                dff['location_long'], errors='coerce'
            )
            dff = dff.dropna(
                subset=['location_lat', 'location_long']
            )
            
    # Start with the map even if no animal match the filter
    markers = [dl.TileLayer()]
    
    # Display the selected animal, or the first available animal
    if not dff.empty:
        row = index[0] if index else 0
        
        if row >= len(dff):
            row = 0
            
        animal = dff.iloc[row]
        
        markers.append(
            dl.Marker(
                position=[
                    animal['location_lat'],
                    animal['location_long']
                ],
                children=[
                    dl.Tooltip(str(animal['breed'])),
                    dl.Popup([
                        html.H4("Animal Information"),
                        html.P(
                            "Name: " + str(animal['name'])
                        ),
                        html.P(
                            "Breed: " + str(animal['breed'])
                        )
                    ])
                ]
            )
        )
        
    # Return the map to the dashboard
    return dl.Map(
        children= markers,
        center=map_center,
        zoom=10,
        style={
            'width':'100%',
            'height': '350px'
        }
    )

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://uraniummanila-thinkindia-3000.codio.io/proxy/8050/
